# Pydantic AI and Nvidia Build models
PydanticAI is a framework to create applications with AI models. PydanticAI is model agnostic and can use several models. In this lecture we will use Nvidia Builds models as they provide good quota limits (40 requests per minute as of this writing).

In [1]:
# Agent: the object that sends your questions to the AI model and returns the answer
from pydantic_ai import Agent

It imports the Agent class from the pydantic_ai library, which you installed with uv add. After that line, you can write Agent(...) in your code.

An import doesn’t run anything or contact any model. It only makes the name Agent available, like taking a tool out of a toolbox.

Agent is the main object in Pydantic AI. You give it a model (your Nvidia one) and it handles sending your prompts and returning the answers. In your code, agent.run("...") is what sends the question.

Your original code used Agent(model) without this import, so Python had no idea what Agent was. That would have raised NameError: name 'Agent' is not defined.

In [8]:
# OpenAIChatModel: a wrapper that talks to any service using OpenAI's chat interface
from pydantic_ai.models.openai import OpenAIChatModel

# OpenAIProvider: holds the connection details (where to connect and how to log in)
from pydantic_ai.providers.openai import OpenAIProvider

import os  # lets us read environment variables

from dotenv import load_dotenv  # reads key=value pairs from a .env file

# Load the variables from your .env file (e.g. NVIDIA_API=...) into the environment.
# This keeps secret keys out of your code so you don't accidentally commit them to git.
load_dotenv()

# Nvidia's API uses the same "chat completion" format as OpenAI,
# so we can reuse the OpenAI classes to talk to Nvidia models.
model = OpenAIChatModel(
    "nvidia/nemotron-3.5-lightning-30b-a3b",  # model name; any model from build.nvidia.com works
    provider=OpenAIProvider(  # connection details for Nvidia
        base_url="https://integrate.api.nvidia.com/v1",  # Nvidia's API address
        api_key=os.environ["NVIDIA_API"],  # your secret key, read from the environment
    ),
)

# The agent wraps the model. Later you can give it instructions, tools, etc.
agent = Agent(model)

# Send a question to the model and wait for the answer.
# `await` pauses until the model replies. It works directly in a Jupyter notebook;
# in a normal .py script you would need asyncio.run(...) instead (see below).
result = await agent.run("What is the capital of Sweden?", timeout=60)

# result.output holds the model's text answer
print(result.output)

The capital of Sweden is **Stockholm**.


In [10]:
result = await agent.run("冗談を言って", timeout = 120)
result

AgentRunResult(output='お医者さん「お薬は１日３回飲んでください」\n患者「はい！」\nお医者さん「そして、３日後にまた来てください」')

In [3]:
result.output

'The capital of Sweden is **Stockholm**.'

# Update the model on the fly

In [4]:
new_model = OpenAIChatModel(
    "nvidia/nemotron-3.5-bolt-e4b",  # any model from build.nvidia.com
    provider=OpenAIProvider( # connection details to nvidia
        base_url="https://integrate.api.nvidia.com/v1",
        api_key=os.environ["NVIDIA_API"],  
    ),
)

result = await agent.run("Tell me a python joke", model=new_model)
result

AgentRunResult(output="Why did the Python programmer quit his job?\n\n**Because he didn't get enough *iterable* feedback!** 🐍😂")

In [5]:
result = await agent.run(
    "Explain how Pydanticai differs from other AI frameworks?",
    model_settings = {"timeout":60},
)
result.output

'It\'s worth noting up front that **"Pydanticai"** doesn\'t correspond to a widely recognized or established AI framework in the public literature, package indexes, or major tech ecosystems. It\'s possible the name is a:\n\n- **Misspelling** of **Pydantic** (a Python data-validation library) combined with AI context\n- **Blend** of **PyTorch** + **Pydantic** or another framework\n- **Very niche or internal tool** not yet publicly documented\n\nI\'ll cover the most likely interpretations and then outline how actual AI frameworks differ, so you can get the answer you need regardless of the exact intent.\n\n### If you meant **Pydantic** in an AI context\n**Pydantic** is not a training or inference framework. It\'s a library for data validation, settings management, and type enforcement using Python type annotations. In AI workflows, it\'s typically used to:\n- Validate input/output schemas for APIs, prompts, and model responses\n- Ensure config files, dataset columns, and JSON payloads ma

In [6]:
with open("pydanticai.md", "w") as file:
    file.write(result.output)

UnicodeEncodeError: 'charmap' codec can't encode character '\u2192' in position 3277: character maps to <undefined>